In [1]:
import pandas as pd
import numpy as np

# 1. Load the shared weekly demand data
file_path = "/Users/anvitha01/Documents/Creative Component/Data/processed/weekly_demand.csv"
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print(df.head())
print(df["dataset"].value_counts())

Original shape: (423579, 4)
   dataset entity_id  week_start         y
0  walmart      10_1  2010-02-02  40212.84
1  walmart      10_1  2010-02-09  67699.32
2  walmart      10_1  2010-02-16  49748.33
3  walmart      10_1  2010-02-23  33601.22
4  walmart      10_1  2010-03-02  36572.44
dataset
walmart      421570
instacart      2009
Name: count, dtype: int64


In [2]:
# 2. Create a sortable week key for each dataset

df["week_sort"] = np.nan

# Walmart: convert real date strings to datetime
mask_walmart = df["dataset"].str.lower() == "walmart"
df.loc[mask_walmart, "week_sort"] = pd.to_datetime(
    df.loc[mask_walmart, "week_start"]
).astype("int64")

# Instacart: extract numeric week from strings like week_001
mask_instacart = df["dataset"].str.lower() == "instacart"
df.loc[mask_instacart, "week_sort"] = (
    df.loc[mask_instacart, "week_start"]
    .str.extract(r"(\d+)")
    .astype(float)
)

print(df[["dataset", "week_start", "week_sort"]].head(10))

   dataset  week_start     week_sort
0  walmart  2010-02-02  1.265069e+15
1  walmart  2010-02-09  1.265674e+15
2  walmart  2010-02-16  1.266278e+15
3  walmart  2010-02-23  1.266883e+15
4  walmart  2010-03-02  1.267488e+15
5  walmart  2010-03-09  1.268093e+15
6  walmart  2010-03-16  1.268698e+15
7  walmart  2010-03-23  1.269302e+15
8  walmart  2010-03-30  1.269907e+15
9  walmart  2010-04-06  1.270512e+15


In [3]:
# 3. Sort by dataset, entity, and week order
df = df.sort_values(by=["dataset", "entity_id", "week_sort"]).reset_index(drop=True)

print(df.head(20))

      dataset entity_id week_start        y  week_sort
0   instacart         1   week_001  27747.0        NaN
1   instacart         1   week_002  14964.0        NaN
2   instacart         1   week_003   9207.0        NaN
3   instacart         1   week_004   5900.0        NaN
4   instacart         1   week_005   4152.0        NaN
5   instacart         1   week_006   2994.0        NaN
6   instacart         1   week_007   2221.0        NaN
7   instacart         1   week_008   1531.0        NaN
8   instacart         1   week_009    987.0        NaN
9   instacart         1   week_010    754.0        NaN
10  instacart         1   week_011    561.0        NaN
11  instacart         1   week_012    359.0        NaN
12  instacart         1   week_013    301.0        NaN
13  instacart         1   week_014    227.0        NaN
14  instacart         1   week_015     23.0        NaN
15  instacart        10   week_001   4060.0        NaN
16  instacart        10   week_002   1979.0        NaN
17  instac

In [4]:
# 4. Create lag-1 prediction within each dataset + entity
df["y_pred"] = df.groupby(["dataset", "entity_id"])["y"].shift(1)
df["y_true"] = df["y"]
df["model_name"] = "lag1"

print(df.head(20))

      dataset entity_id week_start        y  week_sort   y_pred   y_true  \
0   instacart         1   week_001  27747.0        NaN      NaN  27747.0   
1   instacart         1   week_002  14964.0        NaN  27747.0  14964.0   
2   instacart         1   week_003   9207.0        NaN  14964.0   9207.0   
3   instacart         1   week_004   5900.0        NaN   9207.0   5900.0   
4   instacart         1   week_005   4152.0        NaN   5900.0   4152.0   
5   instacart         1   week_006   2994.0        NaN   4152.0   2994.0   
6   instacart         1   week_007   2221.0        NaN   2994.0   2221.0   
7   instacart         1   week_008   1531.0        NaN   2221.0   1531.0   
8   instacart         1   week_009    987.0        NaN   1531.0    987.0   
9   instacart         1   week_010    754.0        NaN    987.0    754.0   
10  instacart         1   week_011    561.0        NaN    754.0    561.0   
11  instacart         1   week_012    359.0        NaN    561.0    359.0   
12  instacar

In [5]:
# 5. Keep only rows where prediction exists
forecast_lag1 = df.dropna(subset=["y_pred"]).copy()

# Select final columns
forecast_lag1 = forecast_lag1[
    ["dataset", "entity_id", "week_start", "y_true", "y_pred", "model_name"]
]

print("Lag-1 forecast shape:", forecast_lag1.shape)
print(forecast_lag1.head(20))

Lag-1 forecast shape: (420114, 6)
      dataset entity_id week_start   y_true   y_pred model_name
1   instacart         1   week_002  14964.0  27747.0       lag1
2   instacart         1   week_003   9207.0  14964.0       lag1
3   instacart         1   week_004   5900.0   9207.0       lag1
4   instacart         1   week_005   4152.0   5900.0       lag1
5   instacart         1   week_006   2994.0   4152.0       lag1
6   instacart         1   week_007   2221.0   2994.0       lag1
7   instacart         1   week_008   1531.0   2221.0       lag1
8   instacart         1   week_009    987.0   1531.0       lag1
9   instacart         1   week_010    754.0    987.0       lag1
10  instacart         1   week_011    561.0    754.0       lag1
11  instacart         1   week_012    359.0    561.0       lag1
12  instacart         1   week_013    301.0    359.0       lag1
13  instacart         1   week_014    227.0    301.0       lag1
14  instacart         1   week_015     23.0    227.0       lag1
16  in

In [6]:
# 6. Save the lag-1 forecast output
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_lag1.csv"
forecast_lag1.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_lag1.csv


In [8]:
#MAE is good
#RMSE is good
import pandas as pd
import numpy as np

file_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_lag1.csv"
forecast_lag1 = pd.read_csv(file_path)

print("Shape:", forecast_lag1.shape)
print(forecast_lag1.head())

Shape: (420114, 6)
     dataset entity_id week_start   y_true   y_pred model_name
0  instacart         1   week_002  14964.0  27747.0       lag1
1  instacart         1   week_003   9207.0  14964.0       lag1
2  instacart         1   week_004   5900.0   9207.0       lag1
3  instacart         1   week_005   4152.0   5900.0       lag1
4  instacart         1   week_006   2994.0   4152.0       lag1


In [9]:
import pandas as pd
import numpy as np

file_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_lag1.csv"
forecast_lag1 = pd.read_csv(file_path)

print("Shape:", forecast_lag1.shape)
print(forecast_lag1.head())

Shape: (420114, 6)
     dataset entity_id week_start   y_true   y_pred model_name
0  instacart         1   week_002  14964.0  27747.0       lag1
1  instacart         1   week_003   9207.0  14964.0       lag1
2  instacart         1   week_004   5900.0   9207.0       lag1
3  instacart         1   week_005   4152.0   5900.0       lag1
4  instacart         1   week_006   2994.0   4152.0       lag1


In [10]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # avoid division by zero
    nonzero_mask = y_true != 0
    if nonzero_mask.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100

In [11]:
metrics_by_dataset = (
    forecast_lag1.groupby("dataset")
    .apply(lambda g: pd.Series({
        "MAE": mae(g["y_true"], g["y_pred"]),
        "RMSE": rmse(g["y_true"], g["y_pred"]),
        "MAPE": mape(g["y_true"], g["y_pred"]),
        "row_count": len(g)
    }))
    .reset_index()
)

metrics_by_dataset["model_name"] = "lag1"

print(metrics_by_dataset)

     dataset          MAE          RMSE        MAPE  row_count model_name
0  instacart  6623.774400  27502.715940   98.780248     1875.0       lag1
1    walmart  2102.473276   7206.830019  149.241601   418239.0       lag1


In [12]:
metrics_by_dataset = metrics_by_dataset[
    ["dataset", "model_name", "MAE", "RMSE", "MAPE", "row_count"]
]

print(metrics_by_dataset)

     dataset model_name          MAE          RMSE        MAPE  row_count
0  instacart       lag1  6623.774400  27502.715940   98.780248     1875.0
1    walmart       lag1  2102.473276   7206.830019  149.241601   418239.0


In [13]:
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_lag1.csv"
metrics_by_dataset.to_csv(output_path, index=False)

print("Saved metrics to:", output_path)

Saved metrics to: /Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_lag1.csv


In [16]:
#second model: moving average.
import pandas as pd
import numpy as np

file_path = "/Users/anvitha01/Documents/Creative Component/Data/processed/weekly_demand.csv"
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print(df.head())

Original shape: (423579, 4)
   dataset entity_id  week_start         y
0  walmart      10_1  2010-02-02  40212.84
1  walmart      10_1  2010-02-09  67699.32
2  walmart      10_1  2010-02-16  49748.33
3  walmart      10_1  2010-02-23  33601.22
4  walmart      10_1  2010-03-02  36572.44


In [17]:
df["week_sort"] = np.nan

mask_walmart = df["dataset"].str.lower() == "walmart"
df.loc[mask_walmart, "week_sort"] = pd.to_datetime(
    df.loc[mask_walmart, "week_start"]
).astype("int64")

mask_instacart = df["dataset"].str.lower() == "instacart"
df.loc[mask_instacart, "week_sort"] = (
    df.loc[mask_instacart, "week_start"]
    .str.extract(r"(\d+)")
    .astype(float)
)

df = df.sort_values(by=["dataset", "entity_id", "week_sort"]).reset_index(drop=True)
print(df.head(20))

      dataset entity_id week_start        y  week_sort
0   instacart         1   week_001  27747.0        NaN
1   instacart         1   week_002  14964.0        NaN
2   instacart         1   week_003   9207.0        NaN
3   instacart         1   week_004   5900.0        NaN
4   instacart         1   week_005   4152.0        NaN
5   instacart         1   week_006   2994.0        NaN
6   instacart         1   week_007   2221.0        NaN
7   instacart         1   week_008   1531.0        NaN
8   instacart         1   week_009    987.0        NaN
9   instacart         1   week_010    754.0        NaN
10  instacart         1   week_011    561.0        NaN
11  instacart         1   week_012    359.0        NaN
12  instacart         1   week_013    301.0        NaN
13  instacart         1   week_014    227.0        NaN
14  instacart         1   week_015     23.0        NaN
15  instacart        10   week_001   4060.0        NaN
16  instacart        10   week_002   1979.0        NaN
17  instac

In [18]:
df["y_pred"] = (
    df.groupby(["dataset", "entity_id"])["y"]
    .transform(lambda s: s.shift(1).rolling(window=4).mean())
)

df["y_true"] = df["y"]
df["model_name"] = "ma4"

print(df.head(20))

      dataset entity_id week_start        y  week_sort    y_pred   y_true  \
0   instacart         1   week_001  27747.0        NaN       NaN  27747.0   
1   instacart         1   week_002  14964.0        NaN       NaN  14964.0   
2   instacart         1   week_003   9207.0        NaN       NaN   9207.0   
3   instacart         1   week_004   5900.0        NaN       NaN   5900.0   
4   instacart         1   week_005   4152.0        NaN  14454.50   4152.0   
5   instacart         1   week_006   2994.0        NaN   8555.75   2994.0   
6   instacart         1   week_007   2221.0        NaN   5563.25   2221.0   
7   instacart         1   week_008   1531.0        NaN   3816.75   1531.0   
8   instacart         1   week_009    987.0        NaN   2724.50    987.0   
9   instacart         1   week_010    754.0        NaN   1933.25    754.0   
10  instacart         1   week_011    561.0        NaN   1373.25    561.0   
11  instacart         1   week_012    359.0        NaN    958.25    359.0   

In [19]:
forecast_ma4 = df.dropna(subset=["y_pred"]).copy()

forecast_ma4 = forecast_ma4[
    ["dataset", "entity_id", "week_start", "y_true", "y_pred", "model_name"]
]

print("MA4 forecast shape:", forecast_ma4.shape)
print(forecast_ma4.head(20))

MA4 forecast shape: (409909, 6)
      dataset entity_id week_start  y_true    y_pred model_name
4   instacart         1   week_005  4152.0  14454.50        ma4
5   instacart         1   week_006  2994.0   8555.75        ma4
6   instacart         1   week_007  2221.0   5563.25        ma4
7   instacart         1   week_008  1531.0   3816.75        ma4
8   instacart         1   week_009   987.0   2724.50        ma4
9   instacart         1   week_010   754.0   1933.25        ma4
10  instacart         1   week_011   561.0   1373.25        ma4
11  instacart         1   week_012   359.0    958.25        ma4
12  instacart         1   week_013   301.0    665.25        ma4
13  instacart         1   week_014   227.0    493.75        ma4
14  instacart         1   week_015    23.0    362.00        ma4
19  instacart        10   week_005   421.0   1938.75        ma4
20  instacart        10   week_006   306.0   1029.00        ma4
21  instacart        10   week_007   191.0    610.75        ma4
22  inst

In [20]:
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_ma4.csv"
forecast_ma4.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_ma4.csv


In [21]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    nonzero_mask = y_true != 0
    if nonzero_mask.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100

In [22]:
metrics_ma4 = (
    forecast_ma4.groupby("dataset")
    .apply(lambda g: pd.Series({
        "MAE": mae(g["y_true"], g["y_pred"]),
        "RMSE": rmse(g["y_true"], g["y_pred"]),
        "MAPE": mape(g["y_true"], g["y_pred"]),
        "row_count": len(g)
    }))
    .reset_index()
)

metrics_ma4["model_name"] = "ma4"

metrics_ma4 = metrics_ma4[
    ["dataset", "model_name", "MAE", "RMSE", "MAPE", "row_count"]
]

print(metrics_ma4)

     dataset model_name          MAE          RMSE        MAPE  row_count
0  instacart        ma4  8242.208079  27834.344414  273.391292     1473.0
1    walmart        ma4  2155.024813   6518.654025  609.894604   408436.0


In [23]:
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_ma4.csv"
metrics_ma4.to_csv(output_path, index=False)

print("Saved metrics to:", output_path)

Saved metrics to: /Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_ma4.csv


In [25]:
#combine both metrics
import pandas as pd

lag1_path = "/Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_lag1.csv"
ma4_path = "/Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_ma4.csv"

metrics_lag1 = pd.read_csv(lag1_path)
metrics_ma4 = pd.read_csv(ma4_path)

metrics_comparison = pd.concat([metrics_lag1, metrics_ma4], ignore_index=True)

print(metrics_comparison)

     dataset model_name          MAE          RMSE        MAPE  row_count
0  instacart       lag1  6623.774400  27502.715940   98.780248     1875.0
1    walmart       lag1  2102.473276   7206.830019  149.241601   418239.0
2  instacart        ma4  8242.208079  27834.344414  273.391292     1473.0
3    walmart        ma4  2155.024813   6518.654025  609.894604   408436.0


In [26]:
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_comparison.csv"
metrics_comparison.to_csv(output_path, index=False)

print("Saved combined metrics to:", output_path)

Saved combined metrics to: /Users/anvitha01/Documents/Creative Component/outputs/metric/metrics_comparison.csv


In [27]:
lag1_forecast_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_lag1.csv"
ma4_forecast_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_ma4.csv"

forecast_lag1 = pd.read_csv(lag1_forecast_path)
forecast_ma4 = pd.read_csv(ma4_forecast_path)

forecast_comparison = pd.concat([forecast_lag1, forecast_ma4], ignore_index=True)

print(forecast_comparison.shape)
print(forecast_comparison.head())

(830023, 6)
     dataset entity_id week_start   y_true   y_pred model_name
0  instacart         1   week_002  14964.0  27747.0       lag1
1  instacart         1   week_003   9207.0  14964.0       lag1
2  instacart         1   week_004   5900.0   9207.0       lag1
3  instacart         1   week_005   4152.0   5900.0       lag1
4  instacart         1   week_006   2994.0   4152.0       lag1


In [28]:
output_path = "/Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_comparison.csv"
forecast_comparison.to_csv(output_path, index=False)

print("Saved combined forecast results to:", output_path)

Saved combined forecast results to: /Users/anvitha01/Documents/Creative Component/outputs/forecast/forecast_results_comparison.csv
